# 05 — Deployment & Consumption

> **Insiders Loyalty Program** · Customer Segmentation Project

---

Géron (Ch. 2): *"Launch, Monitor and Maintain your system. Your work is not done once you deploy the model."*

This notebook covers:
1. Run the complete pipeline end-to-end
2. Export cluster results to SQLite for BI tools
3. Test the REST API with sample customers
4. Simulate scoring a new customer
5. Architecture overview for production

In [ ]:
from pathlib import Path
import sys
import json
import sqlite3
import subprocess

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from insiders_loyalty_program.config import load_config
config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
print('Ready. Project root:', PROJECT_ROOT)

## 1. Run the Full Pipeline

The production pipeline trains the model and writes:
- `models/model.joblib` — serialised sklearn Pipeline
- `reports/metrics.json` — clustering quality metrics
- `reports/cluster_assignments.csv` — customer → cluster mapping

In [ ]:
from insiders_loyalty_program.models import train_clustering

result = train_clustering(config)

print('Pipeline output:')
for key, val in result.items():
    if key != 'metrics':
        print(f'  {key}: {val}')
print(f'  metrics: {result["metrics"]}')

## 2. Export to SQLite

The cluster assignments are exported to a SQLite database so any BI tool (Metabase, Power BI, Tableau, DBeaver) can connect directly.

In [ ]:
clusters_csv  = PROJECT_ROOT / 'reports' / 'cluster_assignments.csv'
sqlite_path   = PROJECT_ROOT / 'reports' / 'insiders_segments.sqlite'
TABLE_NAME    = 'customer_segments'

df_clusters = pd.read_csv(clusters_csv)

# Label the clusters
if 'monetary' in df_clusters.columns:
    means = df_clusters.groupby('cluster')[['recency_days', 'frequency', 'monetary']].mean()
    means['score'] = means.get('monetary', 0) + means.get('frequency', 0) * 100 - means.get('recency_days', 0)
    rank_map = {}
    for rank, cid in enumerate(means.sort_values('score', ascending=False).index):
        rank_map[cid] = ['INSIDERS', 'Loyalists', 'At-Risk', 'Churned',
                         'Churned2', 'Churned3'][min(rank, 5)]
    df_clusters['segment'] = df_clusters['cluster'].map(rank_map)
    df_clusters['is_insider'] = (df_clusters['segment'] == 'INSIDERS').astype(int)

with sqlite3.connect(sqlite_path) as conn:
    df_clusters.to_sql(TABLE_NAME, conn, if_exists='replace', index=False)

print(f'Exported {len(df_clusters):,} rows to {sqlite_path}')
print(f'Table: {TABLE_NAME}')
print(f'\nSegment counts:')
print(df_clusters['segment'].value_counts().to_string())

In [ ]:
# Verify SQLite export by reading it back
with sqlite3.connect(sqlite_path) as conn:
    df_check = pd.read_sql(f'SELECT * FROM {TABLE_NAME} LIMIT 5', conn)
    n_rows = pd.read_sql(f'SELECT COUNT(*) as n FROM {TABLE_NAME}', conn).iloc[0]['n']

print(f'SQLite table has {n_rows:,} rows.')
display(df_check)

## 3. Score a New Customer

The trained model can segment a new customer based on their RFM profile, without retraining.

In [ ]:
import joblib

model_path = PROJECT_ROOT / 'models' / 'model.joblib'
pipeline   = joblib.load(model_path)
print(f'Model loaded from: {model_path}')
print(f'Pipeline steps: {[name for name, _ in pipeline.steps]}')

In [ ]:
# Define new customers as RFM feature vectors
rfm_features = ['recency_days', 'frequency', 'monetary', 'avg_ticket', 'total_items']
rfm_features = [f for f in rfm_features
                if f in (pipeline.named_steps.get('preprocess', pipeline).feature_names_in_
                         if hasattr(pipeline.named_steps.get('preprocess', pipeline), 'feature_names_in_')
                         else rfm_features)]

new_customers = pd.DataFrame([
    # VIP customer: bought recently, frequently, high value
    {'customer_id': 'NEW_001', 'recency_days': 10,  'frequency': 25, 'monetary': 8500, 'avg_ticket': 340, 'total_items': 500},
    # Average customer: moderate behaviour
    {'customer_id': 'NEW_002', 'recency_days': 90,  'frequency': 5,  'monetary': 800,  'avg_ticket': 160, 'total_items': 60},
    # Churned customer: hasn't bought in a year, low value
    {'customer_id': 'NEW_003', 'recency_days': 350, 'frequency': 2,  'monetary': 150,  'avg_ticket': 75,  'total_items': 10},
])

display(new_customers)

In [ ]:
feature_cols = [c for c in rfm_features if c in new_customers.columns]
X_new        = new_customers[feature_cols].values
clusters     = pipeline.predict(X_new)

new_customers['predicted_cluster'] = clusters

# Map to segment names
if 'monetary' in df_clusters.columns:
    new_customers['predicted_segment'] = new_customers['predicted_cluster'].map(rank_map)

print('Predictions for new customers:')
display(new_customers)

## 4. REST API Test

The FastAPI service exposes a `/predict` endpoint. Start the API with:

```bash
make api
# or: uvicorn src.insiders_loyalty_program.api:app --reload
```

Then test it:

In [ ]:
# This cell runs if the API is already running on port 8000
import urllib.request
import json as _json

API_URL = 'http://localhost:8000'

def api_is_running():
    try:
        urllib.request.urlopen(f'{API_URL}/health', timeout=1)
        return True
    except Exception:
        return False

if api_is_running():
    # Call the predict endpoint
    payload = _json.dumps({'records': new_customers.drop(columns=['predicted_cluster','predicted_segment'], errors='ignore').to_dict('records')}).encode()
    req  = urllib.request.Request(f'{API_URL}/predict', data=payload, headers={'Content-Type': 'application/json'}, method='POST')
    resp = urllib.request.urlopen(req)
    print('API response:', _json.loads(resp.read()))
else:
    print('API is not running. Start it with: make api')
    print('Example curl command:')
    print("""curl -X POST http://localhost:8000/predict \\
  -H 'Content-Type: application/json' \\
  -d '{"records": [{"recency_days": 10, "frequency": 25, "monetary": 8500, "avg_ticket": 340, "total_items": 500}]}'""")

## 5. Delivery Summary

| Output | File | Consumer |
|---|---|---|
| Customer segments | `reports/cluster_assignments.csv` | Data team, CRM export |
| BI-ready database | `reports/insiders_segments.sqlite` | Metabase, Power BI, Tableau |
| Trained model | `models/model.joblib` | REST API, batch scoring |
| Clustering metrics | `reports/metrics.json` | Data team, monitoring |
| REST API | `http://localhost:8000/predict` | CRM, marketing automation |

## 6. Production Architecture

For a production deployment, the recommended architecture is:

```
┌─────────────────────────────────────────────────────────────┐
│  Data Source                                                │
│  (S3 / PostgreSQL / Data Warehouse)                         │
└────────────────────────┬────────────────────────────────────┘
                         │ Monthly batch job
                         ▼
┌─────────────────────────────────────────────────────────────┐
│  Training Pipeline                                          │
│  python -m insiders_loyalty_program.cli train               │
│  Orchestrated by: Airflow / Cron / GitHub Actions           │
└────────────────────────┬────────────────────────────────────┘
                         │
           ┌─────────────┴──────────────┐
           ▼                            ▼
┌─────────────────────┐   ┌────────────────────────────────┐
│  REST API           │   │  BI Dashboard                  │
│  FastAPI + Docker   │   │  Metabase → insiders.sqlite    │
│  Real-time scoring  │   │  Power BI / Tableau            │
└─────────────────────┘   └────────────────────────────────┘
```

**Docker one-command deploy:**
```bash
docker compose up --build
```

---

## Project Complete ✓

| Step | Status |
|---|---|
| Business Understanding | ✓ Notebook 00 |
| Data Understanding | ✓ Notebook 01 |
| Exploratory Analysis | ✓ Notebook 02 |
| Feature Engineering | ✓ Notebook 03 |
| Modelling & Results | ✓ Notebook 04 |
| Deployment | ✓ This notebook |

The Insiders loyalty program has a clear, automated, and reproducible ML pipeline from raw transactions to actionable customer segments.